In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
AI 面试模拟器
功能：
1. 选择岗位
2. 3-2-1 倒计时后开始面试
3. AI 面试官从题库随机出题（文字转语音）
4. 面试者点击麦克风回答问题（百度语音识别）
5. deepseek 大模型 根据面试者的回答生成跟进问题
6. 多轮问答
"""

import os
import time
import random
import threading
import tempfile
import requests
import pandas as pd
import tkinter as tk
from tkinter import ttk, scrolledtext, messagebox
from gtts import gTTS
import pygame
import pyaudio
from aip import AipSpeech

# ==================== 配置 ====================
DEEPSEEK_API_KEY = "sk-ec7ce862682a4ef3bfb00fb93f69e517"
DEEPSEEK_API_URL = "https://api.deepseek.com/v1/chat/completions"

# 题库文件路径
QUESTION_BANK_FILE = "interview_question_bank.csv"

# 岗位列表
JOB_LIST = ["产品经理", "软件工程师", "测试工程师", "设计师", "运营"]

# ========== 百度语音识别密钥 ==========
APP_ID = '122596224'
API_KEY = 'qbiNQNmKzZHIkRXNzvI1B0aT'
SECRET_KEY = 'bKolaySlyT6DaT0gY4UYiyfhTxtvVEN5'
# ====================================

# ==================== 初始化 ====================
pygame.mixer.init()

# ==================== 题库管理 ====================
def load_questions():
    """加载面试题库"""
    if os.path.exists(QUESTION_BANK_FILE):# 检查自定义题库文件是否存在
        df = pd.read_csv(QUESTION_BANK_FILE)# 读取CSV文件为DataFrame
        questions_by_job = {}# 初始化岗位-问题字典
        for job in JOB_LIST:# 遍历所有岗位
            # 筛选对应岗位的问题并转为列表
            job_questions = df[df["job"] == job]["question"].tolist()
            questions_by_job[job] = job_questions if job_questions else []
        return questions_by_job
    else:
        print(f"警告: 找不到 {QUESTION_BANK_FILE}，使用默认题库")
        return get_default_questions()

def get_default_questions():
    """默认题库"""
    return {
        "产品经理": [
            "请介绍一个你主导的产品项目。",
            "你如何进行用户需求调研？",
            "如何定义一个产品的成功指标？",
            "你如何进行竞品分析？",
            "如果用户增长停滞，你会如何分析原因？"
        ],
        "软件工程师": [
            "解释什么是面向对象编程。",
            "什么是多线程？它的优缺点是什么？",
            "解释什么是RESTful API。",
            "什么是数据库索引？",
            "如何设计一个高并发系统？"
        ],
        "测试工程师": [
            "什么是软件测试？",
            "测试用例应该包含哪些要素？",
            "什么是回归测试？",
            "如何编写高质量的测试用例？",
            "什么是自动化测试？"
        ],
        "设计师": [
            "请介绍你的设计流程。",
            "什么是用户体验设计？",
            "如何进行用户研究？",
            "什么是设计系统？",
            "如何进行视觉层级设计？"
        ],
        "运营": [
            "请介绍一次你成功的运营活动。",
            "如何提升用户留存率？",
            "如何进行用户增长？",
            "如何设计会员体系？",
            "如何进行内容运营？"
        ]
    }

# ==================== AI 调用 ====================
def call_deepseek_api(messages):
    """调用 DeepSeek API"""
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json"
    }
    
    data = {
        "model": "deepseek-chat",# 指定调用的模型
        "messages": messages# 对话历史（格式：[{"role": "user/assistant", "content": "内容"},...]）
    }
    
    try:
        response = requests.post(DEEPSEEK_API_URL, headers=headers, json=data, timeout=30)
        response.raise_for_status() # 抛出HTTP错误（如401/500）
        result = response.json()# 解析返回的JSON数据
        return result["choices"][0]["message"]["content"]# 提取AI回复内容
    except Exception as e:
        print(f"API 调用错误: {e}")
        return "抱歉，我暂时无法回答，请重试。"

def generate_question(job, questions_bank):
    """根据岗位随机生成问题"""
    job_questions = questions_bank.get(job, [])# 取对应岗位的问题列表
    if job_questions:
        return random.choice(job_questions) # 随机选一个问题
    return f"请介绍一下你在{job}领域的经验。"

def generate_followup(job, answer, conversation_history):
    """根据回答生成跟进问题"""
    # 构造提示词，要求AI生成针对性追问
    prompt = f"""你是一位经验丰富的技术面试官，正在面试{job}岗位。

候选人的回答是：
{answer}

请根据这个回答，生成1个深入的追问问题，要求：
1. 问题要有针对性，基于候选人回答的内容
2. 问题要能考察候选人的专业深度
3. 只输出问题本身，不要加其他说明

追问问题："""
# 拼接对话历史+新提示词
    messages = conversation_history + [
        {"role": "user", "content": prompt}
    ]
    
    return call_deepseek_api(messages)

def evaluate_answer(job, answer):
    """评估回答"""
    # 构造评分提示词（逻辑同generate_followup）
    prompt = f"""你是一位技术面试官。请根据以下面试岗位和候选人回答，给出一个评分（1-10分），并简要说明理由。

面试岗位：{job}

候选人回答：
{answer}

请按以下格式输出：
评分：X/10
理由：..."""

    messages = [{"role": "user", "content": prompt}]
    return call_deepseek_api(messages)

def generate_interview_summary(self, job, user_answers):
        """生成面试总结评估"""
        answers_text = "\n\n".join([f"回答{i+1}: {ans}" for i, ans in enumerate(user_answers)])
    
        prompt = f"""你是一位技术面试官。请根据以下面试记录，给出一个综合评价。

        面试岗位：{job}

        面试记录：
        {answers_text}

        请按以下格式输出：
        总体评分：X/10
        优点：
        1. ...
        2. ...
        待改进：
        1. ...
        2. ...
        综合评价：..."""

        messages = [{"role": "user", "content": prompt}]
        return call_deepseek_api(messages)
     
# ==================== 语音功能 ====================
class BaiduSpeechRecognizer:
    """百度语音识别器"""
    def __init__(self, app_id, api_key, secret_key):
        self.client = AipSpeech(app_id, api_key, secret_key)# 初始化百度语音识别客户端
        self.sample_rate = 16000 # 采样率（百度要求16000）
        
    def record_and_recognize(self, duration=30):
        """录音并识别"""
        try:
            p = pyaudio.PyAudio()# 初始化
            
            # 查找可用的麦克风
            input_device_index = None
            for i in range(p.get_device_count()):
                dev_info = p.get_device_info_by_index(i)
                if dev_info['maxInputChannels'] > 0:
                    input_device_index = i
                    break
            
            if input_device_index is None:
                print("未找到麦克风")
                return ""
            
            # 录音参数（百度要求：16bit、单声道、16000采样率）
            CHUNK = 1600 # 每次读取的音频块大小
            FORMAT = pyaudio.paInt16 # 采样格式（16位）
            CHANNELS = 1 # 单声道
            RATE = self.sample_rate # 采样率16000
            
            # 打开音频流
            stream = p.open(format=FORMAT,
                           channels=CHANNELS,
                           rate=RATE,
                           input=True,
                           input_device_index=input_device_index,
                           frames_per_buffer=CHUNK)
            
            print(f"请开始回答（最多{duration}秒）...")
            
            # 录音：循环读取音频块，持续duration秒
            frames = []
            for i in range(0, int(RATE / CHUNK * duration)):
                data = stream.read(CHUNK, exception_on_overflow=False)
                frames.append(data)
                
            # 停止录音
            stream.stop_stream()
            stream.close()
            p.terminate()
            
            print("录音完成，正在识别...")
            
            # 转换为音频数据
            audio_data = b''.join(frames)
            
            # 调用百度语音识别
            result = self.client.asr(audio_data, 'pcm', RATE, {
                'dev_pid': 1537,
            })
            
            if result['err_no'] == 0:  # 识别成功
                text = result['result'][0]
                print(f"识别结果: {text}")
                return text
            else:
                print(f"识别失败: {result.get('err_msg', '未知错误')} (错误码: {result['err_no']})")
                return ""
                
        except Exception as e:
            print(f"录音或识别错误: {e}")
            return ""


# ==================== GUI 界面 ====================
class InterviewSimulator:
    def __init__(self):
        self.root = tk.Tk()# 初始化tkinter主窗口
        self.root.title("AI 面试模拟器")
        self.root.geometry("700x600")
        self.root.resizable(True, True)
        
        # 面试状态
        self.current_job = tk.StringVar(value=JOB_LIST[0])
        self.is_interviewing = False
        self.conversation_history = []
        self.questions_bank = load_questions()
        self.current_question = ""
        
        # 百度语音识别器
        self.baidu_recognizer = BaiduSpeechRecognizer(APP_ID, API_KEY, SECRET_KEY)
        
        self.setup_ui()
        
    def setup_ui(self):
        """设置界面"""
        # 标题
        title_label = tk.Label(
            self.root, 
            text="🤖 AI 面试模拟器", 
            font=("Arial", 20, "bold"),
            fg="#2c3e50"
        )
        title_label.pack(pady=10) # 布局：垂直间距10
        
        # 岗位选择区域
        job_frame = tk.Frame(self.root)
        job_frame.pack(pady=10)
        
        tk.Label(job_frame, text="选择岗位:", font=("Arial", 12)).pack(side=tk.LEFT, padx=5)
        
        # 岗位下拉框（只读，不可手动输入）
        job_menu = ttk.Combobox(
            job_frame,
            textvariable=self.current_job,
            values=JOB_LIST,
            state="readonly",
            width=15,
            font=("Arial", 11)
        )
        job_menu.pack(side=tk.LEFT, padx=5)
        
        # 开始面试按钮
        self.start_btn = tk.Button(
            job_frame,
            text="🎤 开始面试",
            command=self.start_interview,
            bg="#27ae60",
            fg="white",
            font=("Arial", 11, "bold"),
            padx=15,
            pady=5
        )
        self.start_btn.pack(side=tk.LEFT, padx=20)
        
        # 对话记录区域
        chat_frame = tk.Frame(self.root)
        chat_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        tk.Label(chat_frame, text="对话记录:", font=("Arial", 10, "bold"), anchor="w").pack(fill=tk.X)
        
         # 滚动文本框（显示对话）
        self.chat_text = scrolledtext.ScrolledText(
            chat_frame,
            wrap=tk.WORD,
            font=("微软雅黑", 10),
            height=15,
            state=tk.DISABLED
        )
        self.chat_text.pack(fill=tk.BOTH, expand=True)
        
        # 设置文本颜色标签
        self.chat_text.tag_config("interviewer", foreground="#2980b9", font=("微软雅黑", 10, "bold"))
        self.chat_text.tag_config("user", foreground="#e67e22", font=("微软雅黑", 10, "bold"))
        self.chat_text.tag_config("system", foreground="#7f8c8d", font=("微软雅黑", 9, "italic"))
        
        # 当前问题显示区域
        question_frame = tk.Frame(self.root)
        question_frame.pack(fill=tk.X, padx=10, pady=5)
        
        tk.Label(question_frame, text="当前问题:", font=("Arial", 10, "bold")).pack(anchor="w")
        
        self.question_label = tk.Label(
            question_frame,
            text="请点击「开始面试」",
            wraplength=650,
            justify=tk.LEFT,
            font=("微软雅黑", 10),
            fg="#2c3e50",
            bg="#ecf0f1",
            padx=10,
            pady=5
        )
        self.question_label.pack(fill=tk.X, pady=5)
        
        # 按钮区域
        button_frame = tk.Frame(self.root)
        button_frame.pack(pady=10)
        
        self.answer_btn = tk.Button(
            button_frame,
            text="🎙️ 回答当前问题",
            command=self.answer_question,
            bg="#3498db",
            fg="white",
            font=("Arial", 11, "bold"),
            padx=20,
            pady=8,
            state=tk.DISABLED
        )
        self.answer_btn.pack(side=tk.LEFT, padx=10)
        
        self.end_btn = tk.Button(
            button_frame,
            text="❌ 结束面试",
            command=self.end_interview,
            bg="#e74c3c",
            fg="white",
            font=("Arial", 10),
            padx=20,
            pady=8,
            state=tk.DISABLED
        )
        self.end_btn.pack(side=tk.LEFT, padx=10)
        
        # 状态栏
        self.status_label = tk.Label(
            self.root,
            text="就绪",
            font=("Arial", 9),
            fg="#2c3e50",
            anchor="w"
        )
        self.status_label.pack(side=tk.LEFT, padx=5)
        
    def add_message(self, speaker, message, tag="system"):
        """向对话记录文本框添加消息（面试官 / 用户 / 系统），自动带时间戳和样式"""
        self.chat_text.config(state=tk.NORMAL)# 临时启用编辑
        
        timestamp = time.strftime("%H:%M:%S")# 当前时间戳（时:分:秒）
        
        if speaker == "面试官":
            self.chat_text.insert(tk.END, f"[{timestamp}] ", "system")
            self.chat_text.insert(tk.END, f"{speaker}: ", "interviewer")
        elif speaker == "你":
            self.chat_text.insert(tk.END, f"[{timestamp}] ", "system")
            self.chat_text.insert(tk.END, f"{speaker}: ", "user")
        else:
            self.chat_text.insert(tk.END, f"[{timestamp}] {speaker}: ", "system")
        
        self.chat_text.insert(tk.END, f"{message}\n\n", "system")
        self.chat_text.see(tk.END)# 自动滚动到最新消息
        self.chat_text.config(state=tk.DISABLED)# 禁用编辑
        
    def update_status(self, message):
        """更新状态栏"""
        self.status_label.config(text=message)
        self.root.update()
        
    def countdown(self, seconds=3):
        """倒计时"""
        self.update_status(f"倒计时 {seconds} 秒...")
        for i in range(seconds, 0, -1):
            self.add_message("系统", f"⏰ {i}...", "system")
            time.sleep(1)
        self.add_message("系统", "🎯 面试开始！", "system")
        
    def start_interview(self):
        """开始面试"""
        if self.is_interviewing:
            messagebox.showinfo("提示", "面试正在进行中")
            return
        
        job = self.current_job.get()
        self.add_message("系统", f"你选择的岗位是：{job}", "system")
        
        self.countdown(3)
        
        self.is_interviewing = True
        self.conversation_history = []
        
        self.start_btn.config(state=tk.DISABLED)
        self.answer_btn.config(state=tk.NORMAL)
        self.end_btn.config(state=tk.NORMAL)
        
        self.ask_next_question()
        
    def ask_next_question(self):
        """问下一个问题"""
        job = self.current_job.get()
        
        if not self.conversation_history:
            self.current_question = generate_question(job, self.questions_bank)
            self.add_message("面试官", self.current_question, "interviewer")
            self.conversation_history.append({"role": "assistant", "content": self.current_question})
        
        self.question_label.config(text=self.current_question)
        
        
        
        self.update_status("等待回答...")
        
    def answer_question(self):
        """回答问题"""
        if not self.is_interviewing:
            messagebox.showinfo("提示", "请先开始面试")
            return
        
        if not self.current_question:
            messagebox.showinfo("提示", "当前没有问题，请等待面试官提问")
            return
        
        self.answer_btn.config(state=tk.DISABLED)
        self.update_status("正在录音，请开始回答...")
        self.add_message("系统", "🎙️ 请开始回答（录音中）...", "system")
        
        def record_and_process():
            answer = self.baidu_recognizer.record_and_recognize(duration=30)
            self.root.after(0, lambda: self.process_answer(answer))
        # 启动后台线程
        threading.Thread(target=record_and_process, daemon=True).start()
        
    def process_answer(self, answer):
        """处理回答"""
        if not answer:
            self.add_message("系统", "未检测到有效回答，请重试", "system")
            self.answer_btn.config(state=tk.NORMAL)
            self.update_status("请重试")
            return
        # 记录回答
        self.add_message("你", answer, "user")
        self.conversation_history.append({"role": "user", "content": answer})
        
        self.update_status("正在生成跟进问题...")
        job = self.current_job.get()
        # 生成跟进问题
        followup = generate_followup(job, answer, self.conversation_history)
        # 更新当前问题为跟进问题
        self.current_question = followup
        self.question_label.config(text=self.current_question)
         # 显示跟进问题
        self.add_message("面试官", followup, "interviewer")
        self.conversation_history.append({"role": "assistant", "content": followup})
        
        
        # self.update_status("正在评估回答...")
        # def evaluate():
        #     score = evaluate_answer(job, answer)
        #     self.root.after(0, lambda: self.add_message("系统", f"📊 评估反馈:\n{score}", "system"))
        
        # threading.Thread(target=evaluate, daemon=True).start()
         # 启用回答按钮，等待下一次回答
        self.answer_btn.config(state=tk.NORMAL)
        self.update_status("等待回答...")
        
    def end_interview(self):
        """结束面试"""
        if not self.is_interviewing:
            return
    
        self.is_interviewing = False
    
        self.start_btn.config(state=tk.NORMAL)
        self.answer_btn.config(state=tk.DISABLED)
        self.end_btn.config(state=tk.DISABLED)
    
        # 显示面试总结（可选）
        self.add_message("系统", "✨ 面试已结束，正在生成综合评价...", "system")
    
        # 在新线程中生成总结评估
        def generate_summary():
            job = self.current_job.get()
            # 提取所有用户回答
            user_answers = []
            for msg in self.conversation_history:
                if msg.get("role") == "user":
                    user_answers.append(msg["content"])
        
            if user_answers:
                summary = generate_interview_summary(job, user_answers)
                self.root.after(0, lambda: self.add_message("系统", f"📊 面试总结:\n{summary}", "system"))
            else:
                self.root.after(0, lambda: self.add_message("系统", "感谢你的参与！", "system"))
    
        threading.Thread(target=generate_summary, daemon=True).start()
    
        self.update_status("面试已结束")
        self.question_label.config(text="面试已结束，可重新开始")
       
    def run(self):
        """运行应用"""
        self.root.protocol("WM_DELETE_WINDOW", self.on_close)
        self.root.mainloop()
        
    def on_close(self):
        """关闭窗口时的处理"""
        if self.is_interviewing:
            if messagebox.askyesno("确认", "面试正在进行中，确定要退出吗？"):
                self.root.destroy()
        else:
            self.root.destroy()

# ==================== 主函数 ====================
def main():
    """主函数"""
    try:
        import pygame
        import gtts
    except ImportError as e:
        print("请先安装必要的库:")
        print("pip install pygame gtts pandas requests pyaudio baidu-aip")
        print(f"错误详情: {e}")
        return
    
    app = InterviewSimulator()
    app.run()

if __name__ == "__main__":
    main()

In [3]:
pip install "numpy<2"


录音完成，正在识别...
识别结果: 为我认为那就更不行了嗯。


Exception in thread Thread-6 (record_and_process):
Traceback (most recent call last):
  File "D:\conda\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "D:\conda\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\11496\AppData\Local\Temp\ipykernel_13092\1529961584.py", line 491, in record_and_process
  File "D:\conda\Lib\tkinter\__init__.py", line 873, in after
    name = self._register(callit)
           ^^^^^^^^^^^^^^^^^^^^^^
  File "D:\conda\Lib\tkinter\__init__.py", line 1604, in _register
    self.tk.createcommand(name, f)
RuntimeError: main thread is not in main loop
Exception ignored in: <function Variable.__del__ at 0x0000028D839DA8E0>
Traceback (most recent call last):
  File "D:\conda\Lib\tkinter\__init__.py", line 414, in __del__
    if self._tk.getboolean(self._tk.call("info", "exists", self._name)):
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: main thread is not 

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: numpy<2 in d:\conda\lib\site-packages (1.26.4)



In [ ]:
pip install gtts

In [ ]:
pip install pygame gtts pandas requests pyaudio baidu-aip